# Projeto Final Mina - Fase 01: Entendimento do Negócio e EDA

## 1. Introdução
A análise de dados é fundamental para a tomada de decisão estratégica em ambientes industriais complexos. Na operação de uma mina, falhas não planejadas de equipamentos pesados geram paradas abruptas que impactam a produção e os custos. Este projeto propõe a construção de um pipeline analítico robusto para processar dados de telemetria e prever falhas antes que elas aconteçam.

## 2. Entendimento do Negócio
A operação utiliza equipamentos pesados, divididos principalmente em **Caminhões Fora de Estrada (CA)** para transporte, e **Pás Escavadeiras (PE)** para carregamento. 
O principal evento a ser evitado é o alarme **Don't Go**, que sinaliza uma falha crítica imediata, resultando em paradas corretivas não planejadas.

### O Custo da Falha (Downtime Real)
Na análise exploratória aprofundada, abandonaremos a premissa estática de "12 horas salvas" e buscaremos a média matemática de downtime que cada tipo de falha crítica (Don't Go) realmente causa, baseando-se no histórico oficial de oficina (Apontamentos).


## 3. Metodologia: EDA (Exploratory Data Analysis)
Nesta seção vamos ingerir a fonte da verdade bruta (pasta `data/`), analisar a integridade, realizar o *data cleaning* para extrair o real cenário da operação.


In [ ]:
import pandas as pd
import glob
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual
sns.set_theme(style="whitegrid")

# Ingestão da Fonte da Verdade (Telemetria)
caminho_base = r'data/raw/Base/datasets/telemetria/*.parquet'
arquivos_telemetria = glob.glob(caminho_base)

# Leitura de todas as colunas essenciais para o EDA estendido
dfs_tel = []
for arquivo in arquivos_telemetria:
    df_temp = pd.read_parquet(arquivo, columns=['TAG', 'Data_Evento', 'Is_Dont_Go', 'Alarme', 'Tipo', 'Nome_Operador_Anon'])
    dfs_tel.append(df_temp)

df_telemetria = pd.concat(dfs_tel, ignore_index=True)

# Ingestão de Apontamentos para cruzamento de Downtime
df_apontamentos = pd.read_parquet(r'data/raw/Base/datasets/apontamentos/desenvolver_apontamentos.parquet')

print(f"Total de registros de telemetria ingeridos: {len(df_telemetria)}")
print(f"Total de arquivos de telemetria lidos: {len(arquivos_telemetria)}")




### 3.1 Inspeção e Correção de Nomenclatura (TAGs)
Nosso primeiro passo de Data Quality é garantir que estamos analisando os equipamentos corretos. Uma inspeção revelou que existem erros de digitação onde alguns dígitos do código de equipamento foram omitidos na base (como `CA5926` em vez de `CA65926`). 
Isso fragmentaria o histórico de estresse/fadiga de uma mesma máquina, diluindo nosso modelo analítico.


In [ ]:
# 1. Levantamento das Tags originais na base de dados (Exatamente como foram registradas)
tags_originais = df_telemetria['TAG'].unique()
print(f"Total de TAGs originais (com erro): {len(tags_originais)}")

# 2. Correção direcionada de Loss of Digits
correcoes = {
    'CA5926': 'CA65926',
    'CA5927': 'CA65927'
}

df_telemetria['TAG_Limpa'] = df_telemetria['TAG'].replace(correcoes)
tags_limpas = df_telemetria['TAG_Limpa'].unique()

print(f"Total de TAGs após correção: {len(tags_limpas)}")
print(f"\nForam tratadas 2 nomenclaturas anômalas, reconsolidando a base para {len(tags_limpas)} equipamentos únicos (Caminhões CA e Pás Escavadeiras PE).")


### 3.2 O "Buraco" na Linha do Tempo
Antes de fazer qualquer Engenharia de Features (Fadiga ao longo do tempo), precisamos provar se a nossa série temporal é contínua ou se os sensores ficaram desligados em algum período. Vamos plotar o volume diário global.


In [ ]:
# Adicionando granularidade de Dia e Mês
df_telemetria['Data_Dia'] = pd.to_datetime(df_telemetria['Data_Evento'].dt.date)
df_telemetria['Mes'] = df_telemetria['Data_Evento'].dt.month
df_telemetria['Dia'] = df_telemetria['Data_Evento'].dt.day

# Agrupando volumetria diária global
vol_global = df_telemetria.groupby('Data_Dia').size().reset_index(name='Volume')

# O PULO DO GATO: Como o dia 31 não existe no dataset, o Pandas apenas ligava o dia 30 de Maio direto no dia 1 de Junho.
# Precisamos forçar a criação de um calendário completo (reindex) para que o dia 31 apareça no eixo X com volume ZERO.
calendario_completo = pd.date_range(start=vol_global['Data_Dia'].min(), end=vol_global['Data_Dia'].max())
vol_global = vol_global.set_index('Data_Dia').reindex(calendario_completo, fill_value=0).reset_index()
vol_global.rename(columns={'index': 'Data_Dia'}, inplace=True)

# Plot da Série Histórica
plt.figure(figsize=(15, 4))
sns.lineplot(data=vol_global, x='Data_Dia', y='Volume', color='red')
plt.title("Volume de Registros de Telemetria por Dia (Mostrando o Apagão de Maio)")
plt.ylabel("Volume de Registros")
plt.xlabel("Data")
plt.show()

print("\nDESCOBERTA ORGÂNICA: Observando a série temporal de 6 meses, fica evidente que há um 'apagão' completo (queda a zero) no dia 31 de Maio. Isso prova que não podemos usar janelas móveis baseadas em quantidade de linhas (row-based).")
print("Em nossa etapa de Feature Engineering, o código deverá ser obrigatoriamente referenciado ao relógio (`pd.Grouper(freq='8H')`). Assim, qualquer janela temporal que cruzar o dia 31 de Maio compreenderá o tempo decorrido sem dados e naturalmente zerará o estresse acumulado.")


### 3.3 Base de Valoração (Downtime Médio por Falha)
A premissa engessada de 12 horas por acerto não reflete a realidade operacional. Para sabermos o verdadeiro ROI, precisamos medir o **Downtime Real**. 
A forma mais justa é: buscar o momento exato em que a máquina acusa um evento `Don't Go`, e calcular quantas horas se passam até ela registrar a classe `Operando` (retorno oficial ao trabalho) nos Apontamentos. O tempo inoperante será a diferença matemática entre a falha catastrófica e a volta à operação.


In [ ]:
# 1. Filtro dos alarmes Críticos Don't Go na telemetria (Ordenados por tempo)
dontgo = df_telemetria[df_telemetria['Is_Dont_Go'] == 1].sort_values('Data_Evento').copy()

# 2. Filtro da base de Apontamentos para registros de retorno ao trabalho ('Operando')
ap_operando = df_apontamentos[df_apontamentos['Classe'] == 'Operando'].sort_values('Inicio').copy()

# Normalização estrita de Datetime para evitar o MergeError (us vs ns)
dontgo['Data_Evento'] = dontgo['Data_Evento'].astype('datetime64[ns]')
ap_operando['Inicio'] = ap_operando['Inicio'].astype('datetime64[ns]')

# 3. Merge asof (Merge por proximidade temporal futura)
# Para cada Don't Go, encontramos o PRIMEIRO registro 'Operando' que ocorreu APÓS o alarme.
downtime_real = pd.merge_asof(
    dontgo, 
    ap_operando[['Tag', 'Inicio']], 
    left_on='Data_Evento', 
    right_on='Inicio', 
    left_by='TAG_Limpa', 
    right_by='Tag', 
    direction='forward'
)

# 4. Cálculo do tempo inoperante total (Do Don't Go até o retorno ao status Operando)
downtime_real['Downtime_Real_H'] = (downtime_real['Inicio'] - downtime_real['Data_Evento']).dt.total_seconds() / 3600

# Agrupamento para valoração (Limpando outliers de máquinas que ficaram meses abandonadas)
tabela_valor = downtime_real[downtime_real['Downtime_Real_H'] < 720].groupby('Alarme')['Downtime_Real_H'].mean().sort_values(ascending=False).to_frame(name='Media_Horas_Inoperante')

print("--- DOWNTIME REAL POR ALARME DON'T GO (Do Alarme Crítico até retorno a Operando) ---")
display(tabela_valor.round(2))


### 3.4 Inferência Baseada em Dados (Quem é CA e quem é PE?)
Não devemos assumir prefixos empiricamente. Vamos cruzar as `TAGs` (na Telemetria) com a coluna `Tipo` (que identifica a categoria da máquina) para extrair do próprio dataset qual é a nossa taxonomia real de frota.


In [ ]:
# Extraindo prefixo das TAGs
df_telemetria['Prefixo'] = df_telemetria['TAG_Limpa'].astype(str).str[:2]

# Agrupando por Prefixo e Tipo para provar o agrupamento
taxonomia = df_telemetria[['Prefixo', 'Tipo']].drop_duplicates().dropna().reset_index(drop=True)
print("--- PROVA DE TAXONOMIA DA FROTA ---")
display(taxonomia)

print("\nA base de dados confirma matematicamente que:")
print("- Prefixos 'CA' pertencem exclusivamente à classe 'Caminhao'.")
print("- Prefixos 'PE' pertencem exclusivamente à classe 'Escavadeira'.")


### 3.5 Distribuição do Target (Apontamentos Mensais)
Antes de procurarmos falhas na Telemetria, precisamos observar nosso objetivo: as quebras no Mundo Real (Base de Apontamentos). Ao visualizar a distribuição mensal das quebras e manutenções, definimos nossa *Baseline* de aprendizado e identificamos se a mina sofreu alguma degradação ou sazonalidade agressiva ao longo dos 6 meses.


In [ ]:
# Adicionando Mês da ocorrência real
ap_down['Mes'] = ap_down['Inicio'].dt.month

# Agrupamento e Plot
plt.figure(figsize=(10, 4))
ax = sns.countplot(data=ap_down, x='Mes', hue='Tipo', palette='viridis')
plt.title("Volume de Manutenções e Paradas Reais por Mês (Apontamentos)")
plt.ylabel("Qtd de Ocorrências (Target)")
plt.xlabel("Mês")
plt.legend(title='Frota')
plt.show()

print("\nCONCLUSÃO: O comportamento de quebras no mundo real é relativamente estável entre as frotas ao longo dos meses, sem saltos abruptos. Portanto, qualquer pico bizarro que virmos na Telemetria será puramente falha de sensor (ruído elétrico), e não um reflexo de manutenções.")


### 3.6 Volumetria Global da Telemetria (Busca Ativa de Anomalias)
Já que as paradas são estáveis, vamos varrer os 6 meses inteiros da Telemetria, separando as Frotas (PE vs CA) para detectar *Sensor Loops*. Se não separássemos, os milhões de registros das escavadeiras ocultariam as anomalias dos caminhões.


In [ ]:
# Separação das frotas
df_ca = df_telemetria[df_telemetria['Prefixo'] == 'CA']
df_pe = df_telemetria[df_telemetria['Prefixo'] == 'PE']

# Agrupamento de volume por Dia/Mês/TAG
vol_pe = df_pe.groupby(['TAG_Limpa', 'Mes', 'Dia']).size().sort_values(ascending=False).reset_index(name='Volume')
vol_ca = df_ca.groupby(['TAG_Limpa', 'Mes', 'Dia']).size().sort_values(ascending=False).reset_index(name='Volume')

print("--- TOP 5 VOLUMETRIAS DIÁRIAS: ESCAVADEIRAS (PE) ---")
display(vol_pe.head(5))

print("\n--- TOP 5 VOLUMETRIAS DIÁRIAS: CAMINHÕES (CA) ---")
display(vol_ca.head(5))

print("\nDESCOBERTAS GLOBAIS:")
print("1. [PE]: PE3798 disparou 1.3 Milhão de registros em 29/06; PE3797 disparou 136 mil em 12/01.")
print("2. [CA]: CA65932 disparou 12.356 registros em 26/03 (triplo da média diária de um caminhão).")
print("Faremos o Deep Dive em cada um nas próximas 3 seções.")


### 3.7 Deep Dive #1: PE3798 — Escavadeira (29/06)
Não podemos apenas ver o volume e descartar. Vamos investigar: quais alarmes dispararam, houve `Don't Go`, e o que a oficina registrou?


In [ ]:
pe_loop = df_telemetria[(df_telemetria['TAG_Limpa'] == 'PE3798') & (df_telemetria['Mes'] == 6) & (df_telemetria['Dia'] == 29)]

print(f"Volume total de registros no dia 29/06: {len(pe_loop):,}")
print("\n--- TOP ALARMES ---")
display(pe_loop['Alarme'].value_counts().head(3))

print(f"\nIs_Dont_Go == 1: {pe_loop['Is_Dont_Go'].sum()}")
print("CONCLUSAO SENSOR: Sinal On/Not Configured oscilou 1.2 milhões de vezes. Is_Dont_Go = ZERO. Ruido de sensor.")

ap_pe3798 = df_apontamentos[(df_apontamentos['Tag'] == 'PE3798') & (df_apontamentos['Inicio'].dt.month == 6) & (df_apontamentos['Inicio'].dt.day == 29)]
print("\n--- APONTAMENTOS (29/06) ---")
display(ap_pe3798[['Inicio', 'Fim', 'Classe']].head(4))
print("CONCLUSAO TARGET: Máquina foi para oficina (Parado) sem Don't Go. Loop gerou parada para conserto de sensor. Expurgo obrigatório.")


### 3.8 Deep Dive #2: PE3797 — Escavadeira (12/01)
A PE3797 apareceu com mais de 136 mil registros no dia 12 de Janeiro. Mesma metodologia de investigação.


In [ ]:
pe_loop2 = df_telemetria[(df_telemetria['TAG_Limpa'] == 'PE3797') & (df_telemetria['Mes'] == 1) & (df_telemetria['Dia'] == 12)]

print(f"Volume total de registros no dia 12/01: {len(pe_loop2):,}")
print("\n--- TOP ALARMES ---")
display(pe_loop2['Alarme'].value_counts().head(3))

print(f"\nIs_Dont_Go == 1: {pe_loop2['Is_Dont_Go'].sum()}")
print("CONCLUSAO SENSOR: Alarmes de torque de motor e velocidade ciclando. Is_Dont_Go = ZERO. Não é quebra catastrófica.")

ap_pe3797 = df_apontamentos[(df_apontamentos['Tag'] == 'PE3797') & (df_apontamentos['Inicio'].dt.month == 1) & (df_apontamentos['Inicio'].dt.day >= 12)]
print("\n--- APONTAMENTOS (Jan >= 12) ---")
display(ap_pe3797[['Inicio', 'Fim', 'Classe']].head(4))
print("CONCLUSAO TARGET: No dia seguinte (13/01) ficou Parada por horas. Loop precedeu parada real, mas sem Don't Go. Ruido de sensor. Expurgo necessário.")


### 3.9 Deep Dive #3: CA65932 — Caminhão (26/03)
O CA65932 registrou 12.356 logs em 26/03, o triplo da média diária de um caminhão. Mesma metodologia.


In [ ]:
ca_loop = df_telemetria[(df_telemetria['TAG_Limpa'] == 'CA65932') & (df_telemetria['Mes'] == 3) & (df_telemetria['Dia'] == 26)]

print(f"Volume total de registros no dia 26/03: {len(ca_loop):,}")
print("\n--- TOP ALARMES ---")
display(ca_loop['Alarme'].value_counts().head(3))

print(f"\nIs_Dont_Go == 1: {ca_loop['Is_Dont_Go'].sum()}")
print("CONCLUSAO SENSOR: Solenoide 'Final Drive Bypass' oscilou Active/Inactive ~6.000 vezes cada. Is_Dont_Go = ZERO. Ruido de válvula.")

ap_ca65932 = df_apontamentos[(df_apontamentos['Tag'] == 'CA65932') & (df_apontamentos['Inicio'].dt.month == 3) & (df_apontamentos['Inicio'].dt.day >= 26)]
print("\n--- APONTAMENTOS (Mar >= 26) ---")
display(ap_ca65932[['Inicio', 'Fim', 'Classe']].head(4))
print("CONCLUSAO TARGET: No dia seguinte (27/03) entrou em Manutenção oficial. Loop precedeu manutenção real. Is_Dont_Go = ZERO. Não é falha crítica. Expurgo necessário.")


### 3.10 Expurgo Cirúrgico Final
Com as 3 anomalias investigadas e confirmadas como **ruído de sensor sem Don't Go**, aplicamos o expurgo cirúrgico. Os dias específicos são removidos para evitar que o modelo aprenda um **falso padrão**.


In [ ]:
df_telemetria_tratada = df_telemetria[~((df_telemetria['TAG_Limpa'] == 'PE3798') & (df_telemetria['Mes'] == 6) & (df_telemetria['Dia'] == 29))]
df_telemetria_tratada = df_telemetria_tratada[~((df_telemetria_tratada['TAG_Limpa'] == 'PE3797') & (df_telemetria_tratada['Mes'] == 1) & (df_telemetria_tratada['Dia'] == 12))]
df_telemetria_tratada = df_telemetria_tratada[~((df_telemetria_tratada['TAG_Limpa'] == 'CA65932') & (df_telemetria_tratada['Mes'] == 3) & (df_telemetria_tratada['Dia'] == 26))]

print(f"Registros antes do expurgo: {len(df_telemetria):,}")
print(f"Registros depois do expurgo: {len(df_telemetria_tratada):,}")
print(f"Registros removidos (loops de sensor): {len(df_telemetria) - len(df_telemetria_tratada):,}")


### 3.11 Análise de Operadores
A telemetria contém a coluna `Nome_Operador_Anon` (anonimizada com códigos `OP_XXX`). Antes de ir para o Machine Learning, precisamos investigar dois pontos:
1. Existem erros de nomenclatura nos códigos de operadores? (ex: `OP 001` vs `OP_001`)
2. Existe algum operador que, historicamente, opera equipamentos com **mais ocorrências de `Don't Go`**? Se sim, o operador pode ser uma feature preditiva relevante no Notebook 02.

**Descoberta prévia (via inspeção direta do dataset completo):** A base contém **395 operadores únicos** — todos seguindo o padrão `OP_XXX`, sem erros de nomenclatura. A distribuição de taxa de Don't Go por operador, porém, é fortemente assimétrica: enquanto a média geral é de **0,5%**, o operador `OP_004` apresenta **9,5%** — 19x acima da média. Isso indica que o operador é uma feature preditiva relevante e será incluído no Notebook 02 via *target encoding* (taxa histórica de Don't Go por operador).


In [ ]:
# 1. Inspeção de Nomenclatura dos Operadores
operadores_unicos = df_telemetria['Nome_Operador_Anon'].unique()
print(f"Total de operadores únicos: {len(operadores_unicos)}")
print(f"Valores nulos: {df_telemetria['Nome_Operador_Anon'].isna().sum()}")

# Verificando padrão esperado (OP_XXX)
import re
op_fora_padrao = [op for op in operadores_unicos if not re.match(r'^OP_\d+$', str(op))]
print(f"\nOperadores fora do padrão 'OP_XXX': {len(op_fora_padrao)}")
if op_fora_padrao:
    print(op_fora_padrao[:10])
else:
    print("Todos os operadores seguem o padrão. Nenhum erro de nomenclatura detectado.")

# 2. Ranking de Operadores por Taxa de Don't Go
dontgo_por_op = df_telemetria.groupby('Nome_Operador_Anon').agg(
    total_registros=('Is_Dont_Go', 'count'),
    total_dontgo=('Is_Dont_Go', 'sum')
).reset_index()
dontgo_por_op['taxa_dontgo'] = dontgo_por_op['total_dontgo'] / dontgo_por_op['total_registros']
dontgo_por_op = dontgo_por_op[dontgo_por_op['total_registros'] > 500]  # Filtro de mínimo de exposição
dontgo_por_op = dontgo_por_op.sort_values('taxa_dontgo', ascending=False)

print("\n--- TOP 15 OPERADORES COM MAIOR TAXA DE DON'T GO ---")
display(dontgo_por_op.head(15))

# 3. Plot visual
plt.figure(figsize=(12, 5))
sns.barplot(data=dontgo_por_op.head(15), y='Nome_Operador_Anon', x='taxa_dontgo', palette='rocket')
plt.title("Top 15 Operadores por Taxa de Don't Go")
plt.xlabel("Taxa de Don't Go (Don't Go / Total de Registros)")
plt.ylabel("Operador")
plt.show()

print("\nCONCLUSÃO:")
print(f"Total de operadores na base completa (6 meses): {len(operadores_unicos)}")
print("Nomenclatura: TODOS seguem o padrão OP_XXX. Nenhum erro de digitação detectado.")
print("\nDistribuição de taxa por operador (com > 500 registros):")
print(dontgo_por_op['taxa_dontgo'].describe().round(4))
print("\nO operador é uma feature RELEVANTE: a distribuição é fortemente assimétrica.")
print("Média geral: ~0,5% de Don't Go por registro.")
print("OP_004 (topo do ranking): ~9,5% — 19x acima da média.")
print("\nDecisão: 'Nome_Operador_Anon' entrará no Notebook 02 via Target Encoding")
print("(taxa histórica de Don't Go por operador, calculada apenas no bloco de treino para evitar Data Leakage).")
